In [ ]:
# === Setup ===
# Runtime: <1 minute fast, <2 minutes full on a typical CPU (estimate).
# Hardware: CPU ok; no GPU required.
# Network: none; all datasets are generated locally.
# Competition-safe: general profile; check the actual contest package/data policy.
# Cẩm nang P08: NumPy, pandas, sklearn, Matplotlib, joblib; no package installation.
import os
import random
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
random.seed(42)
np.random.seed(42)
FAST = os.environ.get('OAI_FAST_MODE', '0') == '1'
rng = np.random.default_rng(42)
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)


# Ensembling — Averaging, Blending và Stacking

Reference thực hành. Dự đoán trước mỗi experiment, rồi ghi Result → Observation → Why.

## Data → EDA

Cùng train/validation split cho mọi phương án. Test không cung cấp nhãn cho bất kỳ bước lựa chọn nào. Logistic và forest có cách học khác nhau; điều đó chưa đảm bảo ensemble thắng.

In [ ]:
from time import perf_counter
from sklearn.base import clone
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, log_loss

n_labeled = 400 if FAST else 800
all_X, all_y = make_classification(n_samples=n_labeled + 80, n_features=8,
    n_informative=5, n_redundant=1, flip_y=0.08, class_sep=0.9, random_state=42)
X, y = all_X[:n_labeled], all_y[:n_labeled]
X_test = all_X[n_labeled:]
del all_X, all_y
test_ids = np.array([f'test_{i}' for i in range(len(X_test))])
train_idx, valid_idx = train_test_split(np.arange(len(y)), test_size=0.25,
                                       random_state=42, stratify=y)
Xtr, ytr, Xva, yva = X[train_idx], y[train_idx], X[valid_idx], y[valid_idx]
assert set(train_idx).isdisjoint(valid_idx)
base = [make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, random_state=42)),
        RandomForestClassifier(n_estimators=30 if FAST else 80, max_depth=6,
                               min_samples_leaf=3, random_state=42, n_jobs=1)]

def probabilities(model, inputs):
    """Return class-1 probability (n,) after checking the binary class order."""
    assert model.classes_.tolist() == [0, 1]
    p = model.predict_proba(inputs)
    assert p.shape == (len(inputs), 2)
    assert np.allclose(p.sum(axis=1), 1)
    return p[:, 1]

print('train/validation rows:', len(train_idx), len(valid_idx))
print('training class counts:', np.bincount(ytr))


## Preprocess → Model → Train

Preprocessing logistic nằm trong pipeline. Các models dưới chỉ fit outer-train; outer-validation giữ riêng để so sánh.

In [ ]:
full_models = [clone(est).fit(Xtr, ytr) for est in base]
full_valid = np.column_stack([probabilities(m, Xva) for m in full_models])


### Stacking — OOF trong outer-train

Mỗi hàng Z được dự đoán bởi model không học hàng đó. Meta-model chỉ thấy ytr. Không dùng outer-validation để sinh meta-feature hoặc fit meta-model.

In [ ]:
oof = np.full((len(ytr), len(base)), np.nan)
coverage = np.zeros(len(ytr), dtype=int)
fold_id = np.full(len(ytr), -1)
for fold, (inner_tr, inner_va) in enumerate(StratifiedKFold(3, shuffle=True, random_state=42).split(Xtr, ytr)):
    assert set(inner_tr).isdisjoint(inner_va)
    for j, est in enumerate(base):
        fitted = clone(est).fit(Xtr[inner_tr], ytr[inner_tr])
        oof[inner_va, j] = probabilities(fitted, Xtr[inner_va])
    coverage[inner_va] += 1
    fold_id[inner_va] = fold
assert np.all(coverage == 1) and np.isfinite(oof).all()
meta = LogisticRegression(C=1.0, max_iter=500, random_state=42).fit(oof, ytr)
assert meta.classes_.tolist() == [0, 1]
pd.DataFrame({'row': train_idx, 'fold': fold_id, 'p_lr': oof[:, 0],
              'p_rf': oof[:, 1]}).to_csv(OUT / 'ensemble_oof_solution.csv', index=False)


### Blending — Một inner hold-out

Giữ blend models fit trên inner-train. Chọn weight bằng log loss trên blend set; không refit base sau đó để tránh đổi phân phối input mà weight đã học.

In [ ]:
inner_tr, blend_idx = train_test_split(np.arange(len(ytr)), test_size=0.25,
                                       random_state=43, stratify=ytr)
assert set(inner_tr).isdisjoint(blend_idx)
blend_models = [clone(est).fit(Xtr[inner_tr], ytr[inner_tr]) for est in base]
blend_p = np.column_stack([probabilities(m, Xtr[blend_idx]) for m in blend_models])
weight_rows = []
for weight in [0.0, 0.25, 0.5, 0.75, 1.0]:
    p1 = weight * blend_p[:, 0] + (1 - weight) * blend_p[:, 1]
    weight_rows.append({'weight_lr': weight,
                        'log_loss': log_loss(ytr[blend_idx], np.column_stack([1-p1, p1]), labels=[0, 1])})
weight = min(weight_rows, key=lambda row: row['log_loss'])['weight_lr']
print('inner blending weights:')
print(pd.DataFrame(weight_rows).to_string(index=False))


## Evaluate

**Hypothesis:** diversity của lỗi có thể giúp gộp, nhưng model yếu cũng có thể làm giảm F1. **Result:** so cả score và latency. **Observation/Why:** ghi phương án thắng thực tế, không ép stack/mean đứng đầu.

In [ ]:
def infer(method, inputs):
    """Return class-1 probability (n,) from the exact fitted candidate."""
    if method == 'logistic':
        return probabilities(full_models[0], inputs)
    if method == 'forest':
        return probabilities(full_models[1], inputs)
    if method == 'mean':
        return np.column_stack([probabilities(m, inputs) for m in full_models]).mean(axis=1)
    if method == 'blend':
        p = np.column_stack([probabilities(m, inputs) for m in blend_models])
        return weight * p[:, 0] + (1 - weight) * p[:, 1]
    if method == 'stack':
        z = np.column_stack([probabilities(m, inputs) for m in full_models])
        return probabilities(meta, z)
    raise ValueError('Unknown method')

methods = ['logistic', 'forest', 'mean', 'blend', 'stack']
rows = []
for method in methods:
    start = perf_counter()
    p = infer(method, Xva)
    elapsed = perf_counter() - start
    score = f1_score(yva, (p >= 0.5).astype(int), labels=[0, 1], average='macro', zero_division=0)
    rows.append({'method': method, 'macro_f1': score, 'elapsed_seconds': elapsed})
    # Timing varies; score and predictions should reproduce.
    print(f'{method}: macro_f1={score:.6f}')
    print(f'{method} inference seconds: {elapsed:.6f}')
    assert np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()
comparison = pd.DataFrame(rows)
comparison.to_csv(OUT / 'ensemble_comparison_solution.csv', index=False)
joint_error = np.mean(((full_valid[:, 0] >= 0.5) != yva) & ((full_valid[:, 1] >= 0.5) != yva))
print('joint error rate:', joint_error)
# WHY: deterministic tie-break favors fewer base models, then declared method order.
cost = {'logistic': 1, 'forest': 1, 'mean': 2, 'blend': 2, 'stack': 2}
selected = min(rows, key=lambda r: (-r['macro_f1'], cost[r['method']], methods.index(r['method'])))['method']
print('selected:', selected)


## Submit

Dùng đúng candidate vừa được đánh giá; giữ cả preprocessing/weight/meta nhất quán. Lab chủ đích không refit thêm trên validation. Điểm lựa chọn là development score, không phải held-out final score.

In [ ]:
test_pred = (infer(selected, X_test) >= 0.5).astype(int)
config = {'seed': 42, 'selected': selected, 'threshold': 0.5,
          'train_idx': train_idx.tolist(), 'valid_idx': valid_idx.tolist(),
          'fast_mode': FAST, 'refit_on_validation': False}
config['blend_weight_lr'] = weight
config['inner_train_idx'] = train_idx[inner_tr].tolist()
config['blend_idx'] = train_idx[blend_idx].tolist()
(OUT / 'ensemble_config_solution.json').write_text(json.dumps(config, indent=2), encoding='utf-8')


In [ ]:
# WHY: validate the file read from disk, not only the in-memory frame.
submission = pd.DataFrame({'id': test_ids, 'label': test_pred})
assert submission.columns.tolist() == ['id', 'label']
assert len(submission) == len(test_ids)
assert submission['id'].is_unique
assert submission['label'].isin([0, 1]).all()
submission.to_csv(OUT / 'submission_solution.csv', index=False)
reloaded = pd.read_csv(OUT / 'submission_solution.csv')
assert reloaded['id'].tolist() == list(test_ids)
assert reloaded['label'].tolist() == list(test_pred)
print('submission rows:', len(reloaded))


## Postmortem

Ghi điểm, lỗi chung, latency và chi phí số lần fit. Nếu single model thắng thì giữ nó là kết luận hợp lệ. Chuyển sang group data cần đổi cả inner/outer split, không chỉ outer-validation.